# Economics-Aware Causal Marketing Targeting
## Interactive Research Walkthrough: From Response Prediction to Robust Policy Learning

**Course**: Undergraduate Senior Thesis (CSE / BBA 400), BRAC University  
**Author**: Interdisciplinary Research Team  
**Target Outlets**: *Journal of Marketing Analytics*, *Electronic Commerce Research and Applications*

---
### Overview
This notebook demonstrates the end-to-end empirical pipeline:
1. **Pre-treatment Feature Audit & Randomization Checks** (Zero leakage, SMD < 0.05, Omnibus AUC ~ 0.50)
2. **Conventional Response Prediction vs Causal Uplift (CATE)**
3. **Zero-Inflated Incremental Spend Modeling (Two-Stage Hurdle)**
4. **Budget-Constrained Knapsack Targeting**
5. **Scenario-Based Minimax Regret and Jaccard Policy Stability**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.loader import prepare_hillstrom_splits
from src.data.diagnostics import run_balance_audit
from src.models.response_baselines import ResponseBaseline
from src.models.meta_learners import TLearner, XLearner
from src.models.doubly_robust import DoublyRobustLearner
from src.models.revenue_models import HurdleRevenueUplift
from src.economics.utility import compute_incremental_utility
from src.evaluation.metrics import compute_uplift_curve, compute_qini_score, estimate_net_economic_value

print('All modules loaded successfully!')

## 1. Randomization Verification & Empirical Outcomes

In [ ]:
diag = run_balance_audit(str(project_root / 'data/raw/hillstrom.csv'))
print('Omnibus Randomization Test AUC:', round(diag['omnibus_auc'], 4))
display(diag['outcome_df'])
display(diag['ate_df'])

## 2. Load Leakage-Controlled Splits and Train Models

In [ ]:
splits = prepare_hillstrom_splits(str(project_root / 'data/raw/hillstrom.csv'))
mask_tr = splits.T_train.isin([0, 1])
mask_te = splits.T_test.isin([0, 1])

X_tr, T_tr = splits.X_train[mask_tr], splits.T_train[mask_tr]
y_conv_tr, y_spend_tr = splits.y_conv_train[mask_tr], splits.y_spend_train[mask_tr]

X_te, T_te = splits.X_test[mask_te], splits.T_test[mask_te]
y_conv_te, y_spend_te = splits.y_conv_test[mask_te], splits.y_spend_test[mask_te]

# 1. Response Baseline
resp_model = ResponseBaseline('xgboost').fit(X_tr, y_conv_tr)
p_resp = resp_model.predict_proba(X_te)

# 2. Causal Uplift (T-Learner)
uplift_model = TLearner(is_classification=True).fit(X_tr, T_tr, y_conv_tr)
tau_uplift = uplift_model.predict_cate(X_te)

# 3. Hurdle Incremental Revenue
hurdle_model = HurdleRevenueUplift().fit(X_tr, T_tr, y_conv_tr, y_spend_tr)
rev_uplift = hurdle_model.predict_incremental_revenue(X_te)

print('Models successfully trained and scored on test set!')

## 3. Discrepancy Between Response and Uplift Targeting (Hypothesis H1)

In [ ]:
from scipy import stats
rho, pval = stats.spearmanr(p_resp, tau_uplift)
print(f'Spearman rank correlation: rho = {rho:.4f} (p = {pval:.2e})')

plt.figure(figsize=(7, 5))
plt.scatter(p_resp, tau_uplift, alpha=0.15, color='#2b5c8f')
plt.title('Predicted Response Probability vs Estimated Causal Uplift')
plt.xlabel('Conventional Response Probability P(Y=1 | X)')
plt.ylabel('Estimated Treatment Effect CATE tau(X)')
plt.axhline(0, color='gray', linestyle='--')
plt.show()

## 4. Cumulative Gain Curves & Qini Evaluation

In [ ]:
fracs, qini_resp, rnd = compute_uplift_curve(y_conv_te.values, T_te.values, p_resp)
_, qini_uplift, _ = compute_uplift_curve(y_conv_te.values, T_te.values, tau_uplift)
_, qini_rev, _ = compute_uplift_curve(y_conv_te.values, T_te.values, rev_uplift)

plt.figure(figsize=(9, 6))
plt.plot(fracs * 100, rnd, '--', color='gray', label='Random Targeting')
plt.plot(fracs * 100, qini_resp, label=f'Response Model (Qini: {compute_qini_score(y_conv_te.values, T_te.values, p_resp):.2f})')
plt.plot(fracs * 100, qini_uplift, label=f'T-Learner Uplift (Qini: {compute_qini_score(y_conv_te.values, T_te.values, tau_uplift):.2f})')
plt.plot(fracs * 100, qini_rev, label=f'Hurdle Revenue (Qini: {compute_qini_score(y_conv_te.values, T_te.values, rev_uplift):.2f})')
plt.xlabel('Targeting Coverage (%)')
plt.ylabel('Cumulative Incremental Conversions')
plt.title('Cumulative Uplift Curves on Held-Out Randomized Test Set')
plt.legend()
plt.show()